In [0]:
#Tratamento dos dados salvos na camada bronze
from pyspark.sql.functions import col, to_timestamp, current_timestamp, array, struct, lit, explode_outer

CATALOGO = "workspace"
SCHEMA = "exp_dados"

df_bronze = spark.read.table(f"{CATALOGO}.{SCHEMA}.bronze_cotacoes_raw")

# Get the field names from the rates struct
rates_fields = df_bronze.schema["rates"].dataType.fieldNames()

# Create an array of structs with key-value pairs, casting all values to double
key_value_pairs = array(*[struct(lit(field).alias("key"), col(f"rates.{field}").cast("double").alias("value")) for field in rates_fields])

df_silver = df_bronze.select(
    col("base").alias("moeda_origem"),
    col("date").alias("data_cotacao"),
    explode_outer(key_value_pairs).alias("rate_pair"),
    col("success"),
    col("timestamp"),
    col("_data_ingestao"),
    col("_arquivo_origem")
).select(
    col("moeda_origem"),
    col("data_cotacao"),
    col("rate_pair.key").alias("moeda_destino"),
    col("rate_pair.value").alias("valor_cotacao"),
    col("success"),
    col("timestamp"),
    col("_data_ingestao"),
    col("_arquivo_origem")
).filter("moeda_origem IS NOT NULL AND moeda_destino IS NOT NULL")

tabela_silver = f"{CATALOGO}.{SCHEMA}.silver_cotacoes_limpo"
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_silver)
print("Silver concluída!")